## 環境建置與目標賽事定義

本段程式碼的主要目的是初始化 FastF1 套件的運行環境，並定義我們要分析的資料範圍。

啟用快取 (Cache) 機制：透過建立 ./f1_cache 資料夾，讓 FastF1 將從 FIA 伺服器下載的原始賽事檔案暫存在本地端。這能確保後續在執行「Restart and run all」時，程式會直接讀取硬碟暫存而非重新發送網路請求，除了大幅縮短執行時間，也能避免網路不穩導致的報錯，完美符合期中專題「可重現性」的要求。

定義目標賽事：鎖定 2023 賽季，並挑選出 8 場「純乾地正賽」(如巴林、西班牙、銀石等)。涵蓋了高磨耗與低磨耗的賽道特性，同時排除雨戰與嚴重紅旗中斷的賽事，以確保期末建構最佳化決策模型時，輪胎使用的限制式能夠成立。

In [1]:
import fastf1
import pandas as pd
import os

#settings 避免重跑每次都跟fastf1要資料
cache_dir = './f1_cache'
if not os.path.exists(cache_dir):
    os.makedirs(cache_dir)

# 啟用 FastF1 快取功能
fastf1.Cache.enable_cache(cache_dir)

# 步驟二：定義抓取範圍
target_year = 2023

# 我們挑選的 8 場經典乾地正賽 (填入該分站的回合數 Round Number)
target_races = [
    1,  # R01: Bahrain (高磨耗)
    4,  # R04: Azerbaijan (中低磨耗)
    7,  # R07: Spain (高磨耗)
    9,  # R09: Austria (中低磨耗)
    10, # R10: Great Britain (高磨耗)
    14, # R14: Italy (中低磨耗)
    16, # R16: Japan (高磨耗)
    22  # R22: Abu Dhabi (中低磨耗)
]

# 建立一個空的 List，等一下用來收集這 8 場比賽處理好的 DataFrame
all_laps_data = []

print(f"快取目錄已設定於: {cache_dir}")
print(f"準備抓取 {target_year} 賽季共 {len(target_races)} 場正賽資料...")

快取目錄已設定於: ./f1_cache
準備抓取 2023 賽季共 8 場正賽資料...


## 單圈與氣象數據的提取、合併與匯出

本段程式碼負責向 API 請求資料、進行初步對齊，並輸出為結構化的原始資料表。

關閉微觀遙測 (Telemetry=False)：為避免資料維度過度膨脹導致記憶體溢位，我們關閉了每秒數十筆的油門/煞車訊號下載，將資料控制在「單圈層級」。

數據水平合併 (Concat)：FastF1 的單圈數據 (laps) 與氣象數據 (weather_data) 是分開存放的。我們透過重設索引值 (reset_index)，將每一圈的路面溫度 (TrackTemp) 與氣溫精準對齊到該單圈紀錄上，讓未來的預測模型具備適應環境溫度的能力。

特徵篩選與時間轉換：過濾出後續探索式分析 (EDA) 所需的 16 個關鍵欄位（包含輪胎配方、輪胎壽命、進出站時間等），並將時間物件轉換為純秒數 (LapTime_seconds)，最終將 8 場比賽共約 8,600 筆資料打包匯出成單一 CSV 檔案 (f1_2023_8races_raw.csv)，作為下一步資料清理的原石。

In [3]:
# 步驟三與步驟四：迴圈提取單圈與天氣數據
for r in target_races:
    print(f"正在載入 {target_year} 賽季 R{r:02d} 站正賽資料...")
    # 建立 Session 物件 ('R' 代表 Race 正賽)
    session = fastf1.get_session(target_year, r, 'R')
    
    # 載入資料 (關閉 telemetry 遙測訊號以大幅提升速度，只保留單圈與天氣)
    session.load(telemetry=False, weather=True)
    
    # 取得單圈資料，並強制重設/清除索引以避免合併報錯
    laps = session.laps.reset_index(drop=True)
    
    # 取得該圈時間點的天氣狀況，並強制重設/清除索引
    weather_data = laps.get_weather_data().reset_index(drop=True)
    
    # 將單圈與天氣水平合併 (axis=1)
    laps_with_weather = pd.concat([laps, weather_data], axis=1)
    
    # 新增分站標籤
    laps_with_weather['RoundNumber'] = r
    laps_with_weather['EventName'] = session.event['EventName']
    
    # 挑選我們統整出來的關鍵欄位
    cols_to_keep = [
        'RoundNumber', 'EventName', 'Driver', 'DriverNumber', 
        'LapNumber', 'LapTime', 'Compound', 'TyreLife', 'Stint',
        'PitInTime', 'PitOutTime', 'TrackStatus', 'IsAccurate',
        'TrackTemp', 'AirTemp', 'Rainfall'
    ]
    
    # 過濾出需要的欄位並存入暫存 List
    laps_filtered = laps_with_weather[cols_to_keep].copy()
    all_laps_data.append(laps_filtered)

# 步驟五：合併全賽事資料與匯出
print("\n所有分站載入完畢，正在合併資料...")
raw_df = pd.concat(all_laps_data, ignore_index=True)

# 為了後續畫圖與機器學習方便，將時間格式轉換為純秒數 (Float)
raw_df['LapTime_seconds'] = raw_df['LapTime'].dt.total_seconds()

# 匯出為 CSV 檔案
output_csv = "f1_2023_8races_raw.csv"
raw_df.to_csv(output_csv, index=False)

print(f"資料抓取完成！")
print(f"已成功匯出至 {output_csv}，總共取得 {len(raw_df)} 筆原始單圈資料。")

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


正在載入 2023 賽季 R01 站正賽資料...


req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '14', '55', '44', '18', '63', '77', '10', '23', '22', '2', '20', '21', '27', '24', '4', '31', '16', '81']
core           INFO 	Loading data for Azerbaijan Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


正在載入 2023 賽季 R04 站正賽資料...


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!

...

正在載入 2023 賽季 R16 站正賽資料...


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!

正在載入 2023 賽季 R22 站正賽資料...


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!


所有分站載入完畢，正在合併資料...
資料抓取完成！
已成功匯出至 f1_2023_8races_raw.csv，總共取得 8649 筆原始單圈資料。
